<a href="https://colab.research.google.com/github/Mondin0/data-eng/blob/main/CeL_Data_Eng_Realtime_Producer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Producer de Apache Kafka en Python


In [ ]:
!pip install confluent-kafka faker

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 22.7 MB/s eta 0:00:00


In [ ]:
from confluent_kafka import Producer
from faker import Faker
from datetime import timezone
from random import choice, randint
import json


In [ ]:
# Objeto para generar datos ficticios
fake = Faker()

# Datos de configuración y conexión a Apache Kafka
kafka_config = {
    # URL del servidor, reemplazar por el suyo. Overview -> Kafka API -> Bootstrap server
    'bootstrap.servers': 'cstoobu6igjosi339b60.any.us-east-1.mpx.prd.cloud.redpanda.com:9092',
    'sasl.mechanism': 'SCRAM-SHA-256',
    'security.protocol': 'SASL_SSL',
    'sasl.username': 'usr',
    'sasl.password': 'gpCYTV1Mv0hMZKplGyzWD7uj6O2QBz'
}

# Instanciar un objecto PRODUCER
producer = Producer(kafka_config)

In [ ]:
def generate_fake_log(fake):
    """
    Generar un registro de log ficticio

    Args:
        fake: instancia de Faker
    Returns:
        str: registro de log ficticio
    """
    ip = fake.ipv4()
    tstamp = fake.date_time_between(
        start_date='-1m', end_date='now', tzinfo=timezone.utc)
    request = f"{choice(['GET', 'POST'])} {fake.uri_path()}"
    status_code = choice([200, 401, 403, 404, 500])
    size = randint(2**4, 2**16)
    log = f'{tstamp.strftime("%Y-%m-%d %H:%M:%S")} - {ip} - {request} - {status_code} - {size}'
    return log

def generate_fake_transaction(fake):
    """
    Generar una transacción ficticia de tarjeta de crédito

    Args:
        fake: instancia de Faker
    Returns:
        dict: transacción ficticia
    """
    transaction = {
        'cc_no': fake.credit_card_number(card_type=None),
        'cc_provider': fake.credit_card_provider(card_type=None),
        'cc_expire': fake.credit_card_expire(start='now', end='+10y', date_format='%m/%y'),
        'amount': randint(1, 1000),
        'currency': 'USD',
        'merchant': fake.company(),
        'timestamp': fake.date_time_between(start_date='-6m', end_date='now', tzinfo=timezone.utc).isoformat()
    }
    return transaction

def delivery_callback(err, msg):
    if err is not None:
        print(f'Message failed delivery: {err}')
    else:
        print(f'Message delivered to {msg.topic()}: {msg.value()}')

In [ ]:
generate_fake_log(fake)

'2024-11-22 00:04:12 - 47.95.138.225 - GET blog/list/categories - 403 - 29835'

In [ ]:
# Bucle una cantidad aleatoria de eventos a un topico de kafka
for i in range(randint(5, 20)):
    log_record = generate_fake_log(fake)
    producer.produce(
        'logs', # nombre del topico
        log_record.encode('utf-8'), # codificar el evento a enviar ya que kafka espera msjes codificados
        callback=delivery_callback
    )
    producer.poll(0.5)

Message delivered to logs: b'2024-11-20 00:25:13 - 111.124.27.169 - POST blog/wp-content - 403 - 58087'
Message delivered to logs: b'2024-11-20 00:25:29 - 46.111.234.254 - POST search/tag/wp-content - 404 - 53259'
Message delivered to logs: b'2024-11-20 00:25:22 - 11.107.253.37 - POST tags/wp-content/blog - 401 - 7204'
Message delivered to logs: b'2024-11-20 00:25:17 - 95.59.23.201 - GET tags - 200 - 5272'
Message delivered to logs: b'2024-11-20 00:25:24 - 60.197.157.178 - GET tag/main - 403 - 40489'
Message delivered to logs: b'2024-11-20 00:25:41 - 15.236.83.77 - POST posts - 500 - 24870'
Message delivered to logs: b'2024-11-20 00:25:14 - 61.129.134.67 - GET tag/blog - 200 - 53297'
Message delivered to logs: b'2024-11-20 00:25:14 - 124.23.239.108 - POST tag - 500 - 58703'
Message delivered to logs: b'2024-11-20 00:25:05 - 10.142.182.44 - POST app - 403 - 24595'
Message delivered to logs: b'2024-11-20 00:25:41 - 188.173.37.24 - POST main/list - 200 - 64397'
Message delivered to logs: 

In [ ]:
generate_fake_transaction(fake)

{'cc_no': '4454490462835378',
 'cc_provider': 'American Express',
 'cc_expire': '11/25',
 'amount': 104,
 'currency': 'USD',
 'merchant': 'Hicks Inc',
 'timestamp': '2024-11-19T23:59:15.167224+00:00'}

In [ ]:
# Bucle una cantidad aleatoria de eventos a un topico de kafka
for i in range(randint(1, 10)):
    transaction_record = generate_fake_transaction(fake)
    transaction_record = json.dumps(transaction_record) # Serializacion o codificacion, ya que Kafka espera eso
    producer.produce(
        'transactions',
        value=transaction_record,
        callback=delivery_callback
    )
    producer.poll(0)

Message delivered to transactions: b'{"cc_no": "4468205037612", "cc_provider": "Discover", "cc_expire": "04/26", "amount": 392, "currency": "USD", "merchant": "Hopkins, Barajas and Rodriguez", "timestamp": "2024-11-20T00:02:45.097353+00:00"}'
Message delivered to transactions: b'{"cc_no": "6011338883103341", "cc_provider": "American Express", "cc_expire": "09/31", "amount": 555, "currency": "USD", "merchant": "Nelson-Jones", "timestamp": "2024-11-20T00:01:11.343452+00:00"}'
Message delivered to transactions: b'{"cc_no": "30423101566592", "cc_provider": "Discover", "cc_expire": "02/27", "amount": 437, "currency": "USD", "merchant": "Cannon and Sons", "timestamp": "2024-11-19T23:59:04.447174+00:00"}'
Message delivered to transactions: b'{"cc_no": "371561024111397", "cc_provider": "Mastercard", "cc_expire": "01/33", "amount": 699, "currency": "USD", "merchant": "Mcdonald Inc", "timestamp": "2024-11-19T23:57:23.665092+00:00"}'
Message delivered to transactions: b'{"cc_no": "226869745509098